# Analysis

In [ ]:
!mamba activate mri

In [ ]:
import os
import shutil
from tqdm import tqdm


In [ ]:
# !rm -rf $clean_root

In [ ]:
!ls ${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30001

# Move Raw Patient together

In [ ]:
# Define paths
raw_root = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/raw"  # unzip path
clean_root = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed"


# Loop through subfolders: train, val, test
for split in ["train", "valid", "test"]:
    split_path = os.path.join(raw_root, split)
    
    if not os.path.exists(split_path):
        continue  # Skip if subfolder doesn't exist

    # Loop through patient folders like 30960_2110_3518
    for folder in tqdm(os.listdir(split_path)):
        folder_path = os.path.join(split_path, folder)
        if not os.path.isdir(folder_path):
            continue  # Skip if subfolder doesn't exist

        # Extract patient ID from folder name
        patient_id = folder.split("_")[0]
        dest_folder = os.path.join(clean_root, patient_id)
        os.makedirs(dest_folder, exist_ok=True)

        # Move .nii.gz files into clean/{patient_id}/
        for file in os.listdir(folder_path):
            time_point = file.split("/")[-1].split("_")[0]
            dest_folder_t = os.path.join(dest_folder, time_point)
            os.makedirs(dest_folder_t, exist_ok=True)
                        
            if file.endswith(".nii.gz"):
                src_file = os.path.join(folder_path, file)
                dest_file = os.path.join(dest_folder_t, file)

                # Only copy if the file doesn't already exist at the destination
                if not os.path.exists(dest_file):
                    shutil.copy2(src_file, dest_file)  # Use move() to remove original
#                     print("dest_file=", dest_file)
                    
                    
print("✅ All files moved from train/val/test to clean/ folders by patient.")


In [ ]:
!ls $clean_root/*/*/*


In [ ]:
# Download 3Yr Data, Metadata and csv from https://ida.loni.usc.edu/pages/access/search.jsp?tab=collection&project=ADNI&page=DOWNLOADS&subPage=IMAGE_COLLECTIONS
from glob import glob
import os

root = clean_root


all_files = glob(f"{root}/*/*")     #  patients
print("all_files", all_files[0])
len(all_files)

In [ ]:
!ls ${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30863

# Install and guidance

In [ ]:
# !conda install aramislab::ants   # Download the binary from https://github.com/ANTsX/ANTs
!mamba install -c conda-forge hcc::freesurfer


In [ ]:
# 1. error while loading shared libraries: libITKIOBruker-5.3.so.1: 
# cannot open shared object file: No such file or directory
# Py 3.12
!mamba uninstall  expat -y   # --no-deps
!mamba install -c conda-forge expat=2.1.0 -y
# !mamba install conda-forge::libexpat=2.5.0 -y
!mamba install -c conda-forge itk -y
# !mamba install  -c conda-forge  ants  # Broken


In [ ]:
sudo apt-get install libpng-dev
sudo apt install gettext xterm csh tcsh xorg-dev libncurses5 libffi6 libjpeg62
sudo apt-get install libncurses5 libjpeg62 libtinfo5
# https://surfer.nmr.mgh.harvard.edu/pub/dist/freesurfer/8.0.0-beta/   download freesurfer-linux-centos8
# Install ref to https://surfer.nmr.mgh.harvard.edu/fswiki//FS7_linux

# Test Installation with
freeview -v
recon-all -s bert -all

In [ ]:
!git clone https://github.com/jcreinhold/intensity-normalization
%cd intensity-normalization
!python setup.py install
%cd ..
!ws-normalize 

# Process


In [ ]:
root       = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed"
output_dir = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed"
reference_brain = "${RAW_DATA_DIR}/hao/MNI152_T1_1mm_brain.nii.gz"
# Structure
# root/<subject_id>/<preprocessing>/<date>/<acquisition_id>/<file_name>.nii

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt

# Load the NIfTI image
img = nib.load(reference_brain)
data = img.get_fdata()

# Determine the middle slices for each axis
sagittal_slice = data.shape[0] // 2  # Middle sagittal slice
coronal_slice = data.shape[1] // 2   # Middle coronal slice
axial_slice = data.shape[2] // 2     # Middle axial slice

# Plot the slices
fig, axes = plt.subplots(1, 3, figsize=(6, 2))

# Sagittal view
axes[0].imshow(data[sagittal_slice, :, :], cmap="gray", origin="lower")
axes[0].set_title("Sagittal View")
axes[0].axis("off")

# Coronal view
axes[1].imshow(data[:, coronal_slice, :], cmap="gray", origin="lower")
axes[1].set_title("Coronal View")
axes[1].axis("off")

# Axial view
axes[2].imshow(data[:, :, axial_slice], cmap="gray", origin="lower")
axes[2].set_title("Axial View")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
sample_file = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30003/4954/4954_anat2.nii.gz"

import nibabel as nib
import matplotlib.pyplot as plt

# Load the NIfTI image
img = nib.load(sample_file)
data = img.get_fdata()

# Determine the middle slices for each axis
sagittal_slice = data.shape[0] // 2  # Middle sagittal slice
coronal_slice = data.shape[1] // 2   # Middle coronal slice
axial_slice = data.shape[2] // 2     # Middle axial slice

# Plot the slices
fig, axes = plt.subplots(1, 3, figsize=(6, 2))

# Sagittal view
axes[0].imshow(data[sagittal_slice, :, :], cmap="gray", origin="lower")
axes[0].set_title("Sagittal View")
axes[0].axis("off")

# Coronal view
axes[1].imshow(data[:, coronal_slice, :], cmap="gray", origin="lower")
axes[1].set_title("Coronal View")
axes[1].axis("off")

# Axial view
axes[2].imshow(data[:, :, axial_slice], cmap="gray", origin="lower")
axes[2].set_title("Axial View")
axes[2].axis("off")

plt.tight_layout()
plt.show()




In [ ]:
from tqdm import tqdm
from glob import glob
import os

# subjects  = [f for f in glob(os.path.join(root, "**", "*"), recursive=True) if not f.endswith(".xml")]
subjects  = glob(os.path.join(root, "*"))  #[f for f in glob(os.path.join(root, "*"), recursive=True)]

print(subjects)
print("number of subjects:", len(subjects))

In [ ]:
# Step 1 run in parallel
from concurrent.futures import ThreadPoolExecutor, as_completed


print("Running N4BiasFieldCorrection in Parallel...")


def process_nii(nii, root, subname, output_dir):
    """
    Process a single NIfTI file with N4BiasFieldCorrection.
    """
    input_image = nii
    
    
    if "_step" in nii or "_final" in nii or "_synthseg" in nii:
        return
    
    print("processing:  ", input_image)
    
    step1_output = nii.replace(root, output_dir).rstrip(".nii") + "_step1_biasfield.nii.gz"
    step2_output = step1_output.replace("_step1_biasfield.nii.gz", "_step2_stripped.nii.gz")
    step3_output = step1_output.replace("_step1_biasfield.nii.gz", "_step3_registered.nii.gz")
    step4_output = step1_output.replace("_step1_biasfield.nii.gz", "_synthseg.nii.gz")
    step5_output = step1_output.replace("_step1_biasfield.nii.gz", "_final.nii.gz")

    # List of all steps
    step_outputs = [step1_output, step2_output, step3_output, step4_output, step5_output]

    # Apply the transformation to each step if the file exists
    final_steps = []

    for step_output in step_outputs:
        if os.path.exists(step_output):
            temp = step_output.replace(".nii.gz", "") + ".nii.gz"  # Effectively unchanged unless you add logic here
            shutil.move(step_output, temp)  # You can adjust target path if needed
            final_steps.append(temp)
        else:
            final_steps.append(step_output.replace(".nii.gz", "") + ".nii.gz")

    # Unpack final steps if needed
    step1_output, step2_output, step3_output, step4_output, step5_output = final_steps
    
    parent_dir = os.path.dirname(step1_output)
    os.makedirs(parent_dir, exist_ok=True)
    flag = 0
    
    if not os.path.exists(step1_output):
        os.system(f"${HOME}/hao/software/ants-2.5.4/bin/N4BiasFieldCorrection -d 3 -s 4 -i {input_image} -o {step1_output}")
    else:
        flag = 1
        
    if not os.path.exists(step2_output):
        os.system(f"mri_synthstrip -i {step1_output} -o {step2_output}")  # -g  # Using GPU
    else:
        flag = 2
        
    if not os.path.exists(step3_output):
        os.system(f"${HOME}/hao/software/ants-2.5.4/bin/antsRegistration -d 3 " + \
              f"-n BSpline --shrink-factors 8x4x2x1 --convergence [1000x500x250x100,1e-6,10] --transform Affine[0.1] " + \
              f"--smoothing-sigmas 3x2x1x0vox -m MI[{reference_brain}, {step2_output}, 1,32, Regular, 0.1] " +\
              f"-o [{step3_output +'_transform_prefix'}, {step3_output}]") 
    else:
        flag = 3
        
    if not os.path.exists(step4_output):
        os.system(f"mri_synthseg --i {step3_output} --o {step4_output}")     # --cpu
    else:
        flag = 4
        
    if not os.path.exists(step5_output):
        os.system(f"ws-normalize {step3_output} -o {step5_output}")
    else:
        flag = 5
        
    return f"Processed FLAG = {flag}: {step1_output}"


def process_subject(subject, root, output_dir):
    """
    Process all NIfTI files for a single subject.
    """
#     print("subject = ", subject)
    subname = subject.split("/")[-1]  # Subject name
    niis = glob(subject + "/*/*.nii.gz")
    
#     print("subname=", subject)
#     print("niis=", niis)
    
    results = []
    for nii in niis:
        results.append(process_nii(nii, root, subname, output_dir))
        
    return results


DEBUG = False

# Use ThreadPoolExecutor to process subjects in parallel
max_workers = 2  # min(os.cpu_count()//2, len(subjects))  # Limit workers to the number of CPUs or subjects


with ThreadPoolExecutor(max_workers=max_workers) as executor:
    if DEBUG:
        future_to_subject = {executor.submit(process_subject, sub, root, output_dir): sub for sub in subjects[:2]}
    else:
        future_to_subject = {executor.submit(process_subject, sub, root, output_dir): sub for sub in subjects}
        
        
    for future in tqdm(as_completed(future_to_subject), total=len(future_to_subject)):
        subject = future_to_subject[future]
        try:
            results = future.result()
            for result in results:
                print(result)
                
        except Exception as e:
            print(f"Error processing {subject}: {e}")

print("---------  Done ---------")



In [ ]:
!ls ${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30863/*

In [ ]:
# debug command visualization
import os
verbose = 1
if verbose:
    print("-" * 36)
    print("-" * 36)
            
for sub in tqdm(subjects[:1]):
    subname = sub.split("/")[-1]
    # <subject_id>/<preprocessing>/<date>/<acquisition_id>/<file_name>.nii
    niis = glob(sub + "*/*.nii")
    #     print(subname, ": number of niis:", len(niis))
    for nii in tqdm(niis[:1]):
        
        # Step 1
        step1_output = nii.replace(root, output_dir).rstrip(".nii") + "_step1_biasfield.nii.gz"
        if verbose:
            print("\nRunning N4BiasFieldCorrection...")
#    
        
        parent_dir = os.path.dirname(step1_output)

        # Recursively create all parent directories
        os.makedirs(parent_dir, exist_ok=True)
        
#         os.system(f"${HOME}/hao/software/ants-2.5.4/bin/N4BiasFieldCorrection -d 3 v 1 -s 4  -i {nii} -o {step1_output}")
        print(f"${HOME}/hao/software/ants-2.5.4/bin/N4BiasFieldCorrection -d 3 -s 4  -i {nii} -o {step1_output}")
       
        # Step 2
        if verbose:
            print("\nRunning mri_synthstrip...")
        step2_output = step1_output.replace("_step1_biasfield.nii.gz", "_step2_stripped.nii.gz")
        
        
#         os.system(f"mri_synthstrip -i {step1_output} -o {step2_output}")
        print(f"mri_synthstrip -i {step1_output} -o {step2_output}")
        
        # Step 3 Need a reference
        if verbose:
            print("\nRunning ANTs Registration...")

            
        REFERENCE_IMAGE = None
        step3_output = step1_output.replace("_step1_biasfield.nii.gz", "_step3_registered.nii.gz")
#         os.system(f"${HOME}/hao/software/ants-2.5.4/bin/antsRegistration -d 3 -f {REFERENCE_IMAGE} -m {step2_output} -o {step3_output}")
        print(f"${HOME}/hao/software/ants-2.5.4/bin/antsRegistration -d 3 " + \
              "-n BSpline --shrink-factors 8x4x2x1 --convergence [1000x500x250x100,1e-6,10] --transform Affine[0.1] " + \
              f"--smoothing-sigmas 3x2x1x0vox -m MI[{reference_brain}, {step2_output}, 1,32, Regular, 0.1] " +\
              f"-o [{step3_output +'_transform_prefix'}, {step3_output}]")  # -v 1 # for verbose
    
#             [outputTransformPrefix,<outputWarpedImage>,<outputInverseWarpedImage>]

        
        # Step 4
        if verbose:
            print("\nRunning mri_synthseg...")
        step4_output = step1_output.replace("_step1_biasfield.nii.gz", "_synthseg.nii.gz")
#         os.system(f"mri_synthseg -i {step3_output} -o {step4_output}")
        print(f"mri_synthseg --i {step3_output} --o {step4_output}")     # --cpu
        
        # Step 5
        if verbose:
            print("\nRunning ws-normalize...")
        step5_output = step1_output.replace("_step1_biasfield.nii.gz", "_final.nii.gz")
#         os.system(f"ws-normalize {step4_output} -o {step5_output}")
        print(f"ws-normalize {step3_output} -o {step5_output}")
        
        if verbose:
            print("-" * 36)
            print("-" * 36)

# Visualize the results

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt


nii = "${RAW_DATA_DIR}/hao/ADNI/3Yr/ADNI/036_S_0945/MPR__GradWarp__B1_Correction__N3__Scaled/2008-11-24_10_08_23.0/I130059/ADNI_036_S_0945_MR_MPR__GradWarp__B1_Correction__N3__Scaled_Br_20081211112343845_S60045_I130059.nii"

step1_output = nii.replace(root, output_dir).rstrip(".nii") + "_step1_biasfield.nii.gz"
step2_output = step1_output.replace("_step1_biasfield.nii.gz", "_step2_stripped.nii.gz")
step3_output = step1_output.replace("_step1_biasfield.nii.gz", "_step3_registered.nii.gz")
step4_output = step1_output.replace("_step1_biasfield.nii.gz", "_step4_synthseg.nii.gz")
step5_output = step1_output.replace("_step1_biasfield.nii.gz", "_final.nii.gz")


files = [nii, step1_output, step2_output, step3_output, step4_output, step5_output]
images = [nib.load(f).get_fdata() for f in files]

# Choose a slice index to visualize
slice_index = images[0].shape[2] // 2  # Middle slice in the z-dimension
titles = [
    "Original Image (T1W)", 
    "Step 1: Bias Field Corrected", 
    "Step 2: Skull Stripped", 
    "Step 3: Registered", 
    "Step 4: SynthSeg Processed", 
    "Step 5: Final Output"
]


# Plot the slices side-by-side
fig, axes = plt.subplots(1, len(images), figsize=(15, 5))

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img[:, :, slice_index], cmap="gray")
    ax.axis("off")
    ax.set_title(title, fontsize=10)  # Add a title for each subplot

    
plt.tight_layout()
plt.show()



In [ ]:
for sub in subjects:
    subname = sub.split("/")[-1]
    # <subject_id>/<preprocessing>/<date>/<acquisition_id>/<file_name>.nii
    niis = glob(sub + "/*/*/*/*.nii")
    print(subname, ": number of niis:", len(niis))

In [ ]:
#!/bin/bash

# Input T1W image path
T1W_PATH=$1

# Output directory for results
OUTPUT_DIR=$2

# Step 1: Bias field correction with N4BiasFieldCorrection
echo "Running N4BiasFieldCorrection..."
${HOME}/hao/software/ants-2.5.4/bin/N4BiasFieldCorrection -d 3 v 1 -s 4  -i $T1W_PATH -o $OUTPUT_DIR/T1w_BiasField.nii.gz

# Step 2: Synthetic strip for skull stripping using mri_synthstrip
echo "Running mri_synthstrip..."
mri_synthstrip -i $OUTPUT_DIR/T1w_BiasField.nii.gz -o $OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz

# Step 3: ANTs Registration for image alignment (assuming you have a reference image)
REFERENCE_IMAGE=$3  # Input reference image path
echo "Running ANTs Registration..."
${HOME}/hao/software/ants-2.5.4/bin/antsRegistration -d 3 -f $REFERENCE_IMAGE -m $OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz -o $OUTPUT_DIR/OutputPrefix

# Step 4: Synthetic segmentation using mri_synthseg
echo "Running mri_synthseg..."
mri_synthseg -i $OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz -o $OUTPUT_DIR/SynthSeg_Output.nii.gz

# Step 5: Normalize the image using ws-normalize
echo "Running ws-normalize..."
ws-normalize $OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz -o $OUTPUT_DIR/Normalized_Output.nii.gz

echo "Processing complete. Outputs are located in $OUTPUT_DIR"


## Clean

In [ ]:
import os
import glob

# Root directory
root = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed"

# Define suffixes to look for
suffixes = [
    "_step1_biasfield.nii.gz",
    "_step2_stripped.nii.gz",
    "_step3_registered.nii.gz"
]

# Find all .nii.gz files under root
nii_files = glob.glob(os.path.join(root, "**", "*.nii.gz"), recursive=True)

# Filter and delete matching files
for nii in nii_files:
    for suffix in suffixes:
        if nii.endswith(suffix):
            os.remove(nii)
            print(f"Removed: {nii}")
            # break  # No need to check other suffixes once removed

print("Done")


In [ ]:
import os
import glob
import nibabel as nib
import numpy as np
import scipy.ndimage
from tqdm import tqdm

# Root and output directories
root = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed"
output_root = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/OASISv1.0"

# Target resolution
RESOLUTION = 1.5

# Find synthseg and final files
step4_files = glob.glob(os.path.join(root, "**", "*_synthseg.nii.gz"), recursive=True)
final_files = glob.glob(os.path.join(root, "**", "*_final.nii.gz"), recursive=True)

# Map final files by base name
final_map = {os.path.basename(f).replace("_final.nii.gz", ""): f for f in final_files}

print("Found:", len(step4_files), "synthseg files,", len(final_files), "final files")

# Helper to choose dtype
def choose_dtype(data, path):
    if "synthseg" in path.lower() or "synthseg" in path.lower():
        return data.astype(np.uint8)
    else:
        return data.astype(np.float32)

# Process file pairs
for step4_path in tqdm(step4_files):
    base_name = os.path.basename(step4_path).replace("_synthseg.nii.gz", "")
    if base_name not in final_map:
        continue

    final_path = final_map[base_name]

    for path in [step4_path, final_path]:
        img = nib.load(path)
        data = img.get_fdata()
        affine = img.affine
        header = img.header

        original_spacing = header.get_zooms()[:3]
        original_shape = data.shape

        print(f"\nProcessing: {path}")
        print(f"Original spacing: {original_spacing}")
        print(f"Original shape: {original_shape}")
        print(f"Original min/max: {np.min(data)} / {np.max(data)}")

        # Compute zoom factors for resampling
        zoom_factors = [orig / RESOLUTION for orig in original_spacing]
        # Segmentation labels: nearest neighbor (order=0) to keep discrete label values
        # Intensity images: linear interpolation (order=1)
        is_seg = "synthseg" in os.path.basename(path).lower()
        interp_order = 0 if is_seg else 1
        print(f"Interpolation order: {interp_order} ({'segmentation' if is_seg else 'intensity'})")
        new_data = scipy.ndimage.zoom(data, zoom_factors, order=interp_order)

        # Adjust affine
        new_affine = affine.copy()
        scale_ratio = np.array(original_spacing) / RESOLUTION
        new_affine[:3, :3] = affine[:3, :3] @ np.diag(1 / scale_ratio)

        # Apply dtype optimization
        new_data = choose_dtype(new_data, path)
        print(f"Saved as dtype: {new_data.dtype}")

        # Build output path with consistent name
        rel_path = os.path.relpath(path, root)
        output_path = os.path.join(output_root, rel_path)
        output_path = output_path.replace(".nii.gz", "") + ".nii.gz"
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # Save
        nib.save(nib.Nifti1Image(new_data, new_affine), output_path)

        print(f"Resampled spacing: {[RESOLUTION]*3}")
        print(f"Resampled shape: {new_data.shape}")
        print(f"Saved to: {output_path}")
